In [2]:
import os
checkpoint_dir = "D:/cxr-triage/checkpoints"
for folder in os.listdir(checkpoint_dir):
    full_path = os.path.join(checkpoint_dir, folder)
    if os.path.isdir(full_path):
        for f in os.listdir(full_path):
            print(os.path.join(folder, f))

clahe_320\best_model.pth
clahe_320_logits_fix\best_model.pth
convnext\best_model.pth
focal_loss\best_model.pth
New folder\best_model.pth
New folder\clahe_320
New folder\convnext
New folder\focal_loss


In [3]:
import sys
sys.path.append('D:/cxr-triage')

mods_to_remove = [key for key in sys.modules if 'src' in key]
for mod in mods_to_remove:
    del sys.modules[mod]

import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from tqdm import tqdm
import torchvision.transforms as T
from src.data.dataset import ChestXrayDataset
from src.data.transforms import CLAHETransform
from src.models.densenet import DenseNetModel

LABELS = [
    'Atelectasis', 'Consolidation', 'Infiltration',
    'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis',
    'Effusion', 'Pneumonia', 'Pleural_Thickening',
    'Cardiomegaly', 'Nodule', 'Mass', 'Hernia'
]

CRITICAL = ['Pneumothorax', 'Edema', 'Pneumonia', 'Hernia']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMAGE_ROOT = "F:/X ray dataset/Second Version"
CHECKPOINT_PATH = "D:/cxr-triage/checkpoints/clahe_320_logits_fix/best_model.pth"

def get_transform(image_size=320):
    return T.Compose([
        T.Resize((image_size, image_size)),
        CLAHETransform(clip_limit=2.0, tile_size=8),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

# Load model once
model = DenseNetModel(num_classes=14, pretrained=False).to(DEVICE)
checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Model loaded from epoch {checkpoint['epoch']+1}, AUC {checkpoint['best_auc']:.4f}")

def get_predictions(df, image_size=320):
    dataset = ChestXrayDataset(csv_path=None, image_root=IMAGE_ROOT, transform=get_transform(image_size))
    dataset.df = df
    loader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)
    
    all_logits = []
    all_targets = []
    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Inference"):
            images = images.to(DEVICE)
            with torch.amp.autocast('cuda'):
                logits = model(images)
            all_logits.append(logits.cpu().numpy())
            all_targets.append(targets.numpy())
    
    logits = np.concatenate(all_logits, axis=0)
    targets = np.concatenate(all_targets, axis=0)
    probs = 1 / (1 + np.exp(-logits))  # sigmoid applied here, once, correctly
    return probs, targets

print("Functions ready")

Model loaded from epoch 15, AUC 0.7943
Functions ready


In [4]:
model = DenseNetModel(num_classes=14, pretrained=False).to(DEVICE)
checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Model loaded from epoch {checkpoint['epoch']+1}, AUC {checkpoint['best_auc']:.4f}")

Model loaded from epoch 15, AUC 0.7943


In [5]:
val_df = pd.read_csv('D:/cxr-triage/data/processed/val.csv')
print("Getting validation predictions...")
val_probs, val_targets = get_predictions(val_df, image_size=320)
print(f"Val shape: {val_probs.shape}")

Getting validation predictions...


Inference: 100%|██████████| 2201/2201 [03:35<00:00, 10.19it/s]

Val shape: (17606, 14)


In [9]:
from sklearn.metrics import precision_recall_curve

optimal_thresholds = {}

print(f"\n{'Finding':<22} {'Optimal Threshold':<20} {'Val F1':<10}")
print("-" * 55)

for i, label in enumerate(LABELS):
    precisions, recalls, thresholds = precision_recall_curve(val_targets[:, i], val_probs[:, i])
    f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    optimal_thresholds[label] = float(best_threshold)
    
    print(f"{label:<22} {best_threshold:<20.4f} {f1_scores[best_idx]:<10.4f}")

print("-" * 55)
print("\nThresholds found using VALIDATION set only — no test set leakage")


Finding                Optimal Threshold    Val F1    
-------------------------------------------------------
Atelectasis            0.7031               0.3384    
Consolidation          0.7490               0.1938    
Infiltration           0.6221               0.3657    
Pneumothorax           0.7603               0.2175    
Edema                  0.8818               0.2497    
Emphysema              0.7520               0.1932    
Fibrosis               0.8462               0.1504    
Effusion               0.7397               0.5045    
Pneumonia              0.7671               0.0688    
Pleural_Thickening     0.7441               0.1622    
Cardiomegaly           0.9160               0.3466    
Nodule                 0.6748               0.2088    
Mass                   0.8115               0.2995    
Hernia                 0.8975               0.4571    
-------------------------------------------------------

Thresholds found using VALIDATION set only — no test set leak

In [10]:
test_df = pd.read_csv('D:/cxr-triage/data/processed/test.csv')
print("Getting test predictions...")
test_probs, test_targets = get_predictions(test_df, image_size=320)
print(f"Test shape: {test_probs.shape}")

Getting test predictions...


Inference: 100%|██████████| 3200/3200 [04:56<00:00, 10.78it/s]

Test shape: (25596, 14)


In [12]:
test_df = pd.read_csv('D:/cxr-triage/data/processed/test.csv')
print("Getting test predictions...")
test_probs, test_targets = get_predictions(test_df, image_size=320)
print(f"Test shape: {test_probs.shape}")

# Force float32 to avoid sklearn dtype errors
test_probs = test_probs.astype(np.float32)
test_targets = test_targets.astype(np.float32)

# Apply frozen validation thresholds to test set
test_pred = np.zeros_like(test_probs, dtype=np.float32)
for i, label in enumerate(LABELS):
    test_pred[:, i] = (test_probs[:, i] >= optimal_thresholds[label]).astype(np.float32)

print("\nMicro F1:", f1_score(test_targets, test_pred, average='micro'))
print("Macro F1:", f1_score(test_targets, test_pred, average='macro'))

# Per-class breakdown
print(f"\n{'Finding':<22} {'AUC':<8} {'F1':<8} {'Precision':<12} {'Recall':<8} {'Support':<8}")
print("-" * 75)

for i, label in enumerate(LABELS):
    auc = roc_auc_score(test_targets[:, i], test_probs[:, i])
    f1 = f1_score(test_targets[:, i], test_pred[:, i], zero_division=0)
    prec = precision_score(test_targets[:, i], test_pred[:, i], zero_division=0)
    rec = recall_score(test_targets[:, i], test_pred[:, i], zero_division=0)
    support = int(test_targets[:, i].sum())
    print(f"{label:<22} {auc:<8.4f} {f1:<8.4f} {prec:<12.4f} {rec:<8.4f} {support:<8}")

print("-" * 75)
mean_auc = np.mean([roc_auc_score(test_targets[:, i], test_probs[:, i]) for i in range(14)])
print(f"\nMean Test AUC: {mean_auc:.4f}")

Getting test predictions...


Inference: 100%|██████████| 3200/3200 [04:57<00:00, 10.76it/s]


Test shape: (25596, 14)

Micro F1: 0.34033051535559056
Macro F1: 0.25479132954856

Finding                AUC      F1       Precision    Recall   Support 
---------------------------------------------------------------------------
Atelectasis            0.7225   0.3405   0.2620       0.4861   3279    
Consolidation          0.7210   0.2272   0.1483       0.4848   1815    
Infiltration           0.6761   0.4440   0.3068       0.8032   6112    
Pneumothorax           0.8096   0.3819   0.3188       0.4762   2665    
Edema                  0.8172   0.1971   0.1524       0.2789   925     
Emphysema              0.7996   0.2620   0.1832       0.4602   1093    
Fibrosis               0.7643   0.0891   0.0728       0.1149   435     
Effusion               0.7968   0.4864   0.4091       0.5996   4658    
Pneumonia              0.6569   0.0693   0.0514       0.1063   555     
Pleural_Thickening     0.7256   0.1687   0.1112       0.3491   1143    
Cardiomegaly           0.8799   0.3792   0.4208  

In [ ]:
#############################################################

In [ ]:
train_patients = set(train_df['Patient ID'])
val_patients = set(val_df['Patient ID'])
test_patients = set(test_df['Patient ID'])

print("Train-Val overlap:", len(train_patients & val_patients))
print("Train-Test overlap:", len(train_patients & test_patients))
print("Val-Test overlap:", len(val_patients & test_patients))






NameError: name 'train_df' is not defined

In [16]:
import numpy as np
from sklearn.metrics import f1_score

def find_best_thresholds(y_true, y_probs):
    thresholds = np.zeros(y_true.shape[1])

    for i in range(y_true.shape[1]):
        best_f1 = 0
        best_t = 0.5

        for t in np.linspace(0.05, 0.95, 50):
            preds = (y_probs[:, i] >= t).astype(int)
            f1 = f1_score(y_true[:, i], preds)

            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        thresholds[i] = best_t
        print(f"Class {i}: best threshold={best_t:.2f}, F1={best_f1:.3f}")

    return thresholds

thresholds = find_best_thresholds(y_true, y_probs)

Class 0: best threshold=0.67, F1=0.339
Class 1: best threshold=0.75, F1=0.228
Class 2: best threshold=0.66, F1=0.454
Class 3: best threshold=0.73, F1=0.388
Class 4: best threshold=0.84, F1=0.210
Class 5: best threshold=0.80, F1=0.273
Class 6: best threshold=0.78, F1=0.104
Class 7: best threshold=0.71, F1=0.489
Class 8: best threshold=0.75, F1=0.075
Class 9: best threshold=0.73, F1=0.173
Class 10: best threshold=0.88, F1=0.389
Class 11: best threshold=0.67, F1=0.195
Class 12: best threshold=0.80, F1=0.265
Class 13: best threshold=0.77, F1=0.237


In [17]:
y_pred_new = np.zeros_like(y_probs)

for i in range(y_probs.shape[1]):
    y_pred_new[:, i] = (y_probs[:, i] >= thresholds[i]).astype(int)

In [18]:
from sklearn.metrics import f1_score, roc_auc_score

print("=== AFTER THRESHOLD TUNING ===")

print("Micro F1:", f1_score(y_true, y_pred_new, average='micro'))
print("Macro F1:", f1_score(y_true, y_pred_new, average='macro'))
print("Mean ROC-AUC:", roc_auc_score(y_true, y_probs, average='macro'))

=== AFTER THRESHOLD TUNING ===
Micro F1: 0.3369177272570779
Macro F1: 0.2726762234083188
Mean ROC-AUC: 0.7598929890436705


In [20]:
print([k for k in globals().keys() if "train" in k or "test" in k or "val" in k or "loader" in k])

['__loader__', 'get_val_transforms', 'test_df', 'test_dataset', 'test_loader']


In [21]:
print(y_true.shape, y_probs.shape)

(25596, 14) (25596, 14)


In [22]:
import numpy as np
from sklearn.metrics import f1_score

best_t = 0.5
best_f1 = 0

for t in np.linspace(0.1, 0.9, 100):
    y_pred = (y_probs >= t).astype(int)
    f1 = f1_score(y_true, y_pred, average='macro')

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best threshold:", best_t)
print("Best Macro F1:", best_f1)

Best threshold: 0.7141414141414141
Best Macro F1: 0.25910751876850757


In [23]:
y_pred_final = (y_probs >= best_t).astype(int)

In [24]:
from sklearn.metrics import f1_score, roc_auc_score

print("=== FINAL RESULTS ===")
print("Micro F1:", f1_score(y_true, y_pred_final, average='micro'))
print("Macro F1:", f1_score(y_true, y_pred_final, average='macro'))
print("ROC-AUC:", roc_auc_score(y_true, y_probs, average='macro'))

=== FINAL RESULTS ===
Micro F1: 0.3039075670858076
Macro F1: 0.25910751876850757
ROC-AUC: 0.7598929890436705


In [25]:
print(test_df.head())
print(test_dataset)

        Image Index       Finding Labels  Follow-up #  Patient ID  \
0  00000003_000.png               Hernia            0           3   
1  00000003_001.png               Hernia            1           3   
2  00000003_002.png               Hernia            2           3   
3  00000003_003.png  Hernia|Infiltration            3           3   
4  00000003_004.png               Hernia            4           3   

   Patient Age Patient Gender View Position  OriginalImage[Width  Height]  \
0           81              F            PA                 2582     2991   
1           74              F            PA                 2500     2048   
2           75              F            PA                 2048     2500   
3           76              F            PA                 2698     2991   
4           77              F            PA                 2500     2048   

   OriginalImagePixelSpacing[x     y]  Unnamed: 11  
0                        0.143  0.143          NaN  
1               

In [29]:
import os

base_path = r"F:\X ray dataset"
print(os.listdir(base_path))

['versions', 'Second Version']


In [30]:
import os

print(os.listdir(r"F:\X ray dataset\Second Version"))

['ARXIV_V5_CHESTXRAY.pdf', 'BBox_List_2017.csv', 'Data_Entry_2017.csv', 'FAQ_CHESTXRAY.pdf', 'LOG_CHESTXRAY.pdf', 'README_CHESTXRAY.pdf', 'images_001', 'images_002', 'images_003', 'images_004', 'images_005', 'images_006', 'images_007', 'images_008', 'images_009', 'images_010', 'images_011', 'images_012', 'test_list.txt', 'train_val_list.txt']


In [31]:
import os

base_dir = r"F:\X ray dataset\Second Version"

image_dirs = [
    os.path.join(base_dir, d)
    for d in os.listdir(base_dir)
    if d.startswith("images_")
]

print("Found image folders:", len(image_dirs))
print(image_dirs[:3])

Found image folders: 12
['F:\\X ray dataset\\Second Version\\images_001', 'F:\\X ray dataset\\Second Version\\images_002', 'F:\\X ray dataset\\Second Version\\images_003']


In [32]:
def find_image_path(img_name):
    for d in image_dirs:
        path = os.path.join(d, img_name)
        if os.path.exists(path):
            return path
    return None

In [33]:
import numpy as np
from sklearn.metrics import f1_score

num_classes = y_true.shape[1]
best_thresholds = np.zeros(num_classes)

for c in range(num_classes):
    best_t = 0.5
    best_f1 = 0

    for t in np.linspace(0.1, 0.9, 50):
        preds = (y_probs[:, c] >= t).astype(int)
        f1 = f1_score(y_true[:, c], preds)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    best_thresholds[c] = best_t

print(best_thresholds)

[0.70408163 0.73673469 0.65510204 0.72040816 0.83469388 0.80204082
 0.78571429 0.72040816 0.70408163 0.72040816 0.86734694 0.67142857
 0.80204082 0.75306122]


In [34]:
def topk_enhance(probs, k=2):
    preds = np.zeros_like(probs)

    for i in range(len(probs)):
        topk = probs[i].argsort()[-k:]
        preds[i, topk] = 1

    return preds

In [35]:
final_pred = np.zeros_like(y_probs)

for c in range(y_probs.shape[1]):
    final_pred[:, c] = (y_probs[:, c] >= best_thresholds[c]).astype(int)

# optional boost
final_pred = np.maximum(final_pred, topk_enhance(y_probs, k=2))

In [37]:
y_pred_base = (y_probs >= 0.5).astype(int)

In [40]:
from sklearn.metrics import f1_score

print("BASE Micro F1:", f1_score(y_true, y_pred_base, average='micro'))
print("BASE Macro F1:", f1_score(y_true, y_pred_base, average='macro'))

BASE Micro F1: 0.23382106159031787
BASE Macro F1: 0.2026726335599134


In [41]:
import numpy as np
from sklearn.metrics import f1_score

best_thresholds = np.zeros(y_true.shape[1])

for c in range(y_true.shape[1]):
    best_t = 0.5
    best_f1 = 0

    for t in np.linspace(0.05, 0.95, 40):
        preds = (y_probs[:, c] >= t).astype(int)

        f1 = f1_score(y_true[:, c], preds, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    best_thresholds[c] = best_t

In [42]:
y_pred = np.zeros_like(y_probs)

for c in range(y_probs.shape[1]):
    y_pred[:, c] = (y_probs[:, c] >= best_thresholds[c]).astype(int)

In [43]:
print("Micro F1:", f1_score(y_true, y_pred, average='micro'))
print("Macro F1:", f1_score(y_true, y_pred, average='macro'))

Micro F1: 0.33212513636203683
Macro F1: 0.27214737173142256


In [17]:
import pandas as pd

df = pd.read_csv('D:/cxr-triage/data/processed/train.csv')

labels = [
    'Atelectasis', 'Consolidation', 'Infiltration',
    'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis',
    'Effusion', 'Pneumonia', 'Pleural_Thickening',
    'Cardiomegaly', 'Nodule', 'Mass', 'Hernia'
]

# Test the actual substring logic used in dataset.py against real findings
mismatches = 0
for idx in range(min(2000, len(df))):
    finding = df.iloc[idx]['Finding Labels']
    exact_labels = set(finding.split('|'))
    substring_labels = set(l for l in labels if l in finding)
    
    if exact_labels - {'No Finding'} != substring_labels:
        mismatches += 1
        if mismatches <= 5:
            print(f"Row {idx}: finding='{finding}'")
            print(f"  exact split: {exact_labels}")
            print(f"  substring match: {substring_labels}")
            print()

print(f"\nTotal mismatches out of 2000 checked: {mismatches}")


Total mismatches out of 2000 checked: 0
